In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage,SystemMessage,AIMessage
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

agent = create_agent(
    model=model,
    tools = [],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
config = {"configurable":{"thread_id":"test-1"}}

In [4]:
#Alternative test data
questions= [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config=config)
    print(f"message{response}")
    print(f"messages{len(response['messages'])}")

message{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='ad001330-015e-4a54-b022-4666757eb7df'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.009687566, 'completion_tokens_details': None, 'prompt_time': 0.003977645, 'prompt_tokens_details': None, 'queue_time': 0.161608245, 'total_time': 0.013665211}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eafaa-17ef-73f3-ac73-587957637a75-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
messages2
message{'messages': [HumanMessage(content='What is 10*5?', additional_kwargs={}, response_metadata={}, id='fea2d8c0-7e1c-4c82-a06a-43fad3ad53b0'), AIMessage(con

In [5]:
##token size

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage,SystemMessage
from langchain_core.tools import tool
 
@tool
def search_hostels(city:str)->str:
    '''Serach hostels-retuerns long response to use tokens'''
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

 

llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider='groq'
)

agent = create_agent(
    model=llm,
    tools=[search_hostels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model= llm,
            trigger=("tokens",550),
            keep=("tokens",200)
        ),
    ]
    
    
)

config = {"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [ ]:
cities = ['Paris','Dubai','New york']
for city in cities:
    response = agent.invoke({"messages":[HumanMessage(content=f"find hostels in the {city}")]},
                            config=config)
    tokens = count_tokens(response["messages"])
    print(f"{city}:~{tokens}tokens,{len(response['messages'])}messages")
    print(f"{response["messages"]}")

In [8]:
##FRACTION

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain.messages import HumanMessage,SystemMessage
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware

@tool
def search_hotels(city:str)-> str:
    '''search hotels'''
    return f"Hotels in {city}:: Grand Hotel $350, City Inn $180, Budget Stay $75 "

llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction",0.005),
            keep=("fraction",0.002)
        ),
    ],
    
)

config = {"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    return sum(len(str(m.content))for m in messages) // 4

cities = ["Paris","london","delhi"]
for city in cities:
    response = agent.invoke({"messages":[HumanMessage(content=city)]},config=config)
    tokens = count_tokens(response["messages"])
    fraction = tokens/128000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~51 tokens (0.0398%), 4 msgs
[HumanMessage(content='Paris', additional_kwargs={}, response_metadata={}, id='efb7021f-00e6-469c-b653-e2479f31c0c7'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'a0fc60gbp', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 207, 'total_tokens': 222, 'completion_time': 0.019886552, 'completion_tokens_details': None, 'prompt_time': 0.010944847, 'prompt_tokens_details': None, 'queue_time': 0.160786813, 'total_time': 0.030831399}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eafba-5cbc-7c03-9919-083599f0779c-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'a0fc60gbp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input

In [16]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model

def read_email_tool(email:str)->str:
    """Mock function to Read the email by its Id"""
    return f"Email content for id{email}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to send an emial by its Id"""
    return f"Email sent to {recipient} with {subject}"



In [45]:
from langchain.agents import create_agent
llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)



agent = create_agent(
    model= llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            })
    ]
)



In [46]:
config = {"configurable":{"thread_id":"test-approve"}}
result = agent.invoke({"messages":[HumanMessage(content="send email to john@email.com with subject 'hello' and body 'how are you?'")]},config=config)


In [47]:
result

{'messages': [HumanMessage(content="send email to john@email.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='d5747856-1713-41e8-9355-142ddb9ae3a0'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4a4m28ja7', 'function': {'arguments': '{"body":"how are you?","recipient":"john@email.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 315, 'total_tokens': 347, 'completion_time': 0.079398888, 'completion_tokens_details': None, 'prompt_time': 0.017643532, 'prompt_tokens_details': None, 'queue_time': 0.055705947, 'total_time': 0.09704242}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eaff6-7167-7680-9fdd-36215b682df1-0', tool_calls=[{'name': 'send_email_tool', 'args': {'

In [48]:
##Approved
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )
    print(f"✅ Result: {result['messages'][-1].content}")
    

⏸️ Paused! Approving...
✅ Result: as the subject and how are you? as the body.


In [49]:
llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

agent = create_agent(
    model = llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [50]:
config = {"configurable":{"thread_id":"test_reject"}}
result = agent.invoke({"messages":[HumanMessage(content="send email to nkv@gmail.com with subject'Resume'and body'I will send tommorrow'")]},config=config)

In [51]:
result

{'messages': [HumanMessage(content="send email to nkv@gmail.com with subject'Resume'and body'I will send tommorrow'", additional_kwargs={}, response_metadata={}, id='019961b9-50d5-4a39-92c3-c804be09293a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6xmz6pmgy', 'function': {'arguments': '{"body":"I will send tomorrow","recipient":"nkv@gmail.com","subject":"Resume"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 318, 'total_tokens': 352, 'completion_time': 0.065156331, 'completion_tokens_details': None, 'prompt_time': 0.015634082, 'prompt_tokens_details': None, 'queue_time': 0.161640718, 'total_time': 0.080790413}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eaff7-b86f-75b0-9b91-b8a0a6616f6e-0', tool_calls=[{'name': 'send_email_to

In [53]:
##Approved
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )
    print(f"✅ Result: {result['messages'][-1].content}")
    

⏸️ Paused! Approving...
✅ Result: 


In [54]:
result

{'messages': [HumanMessage(content="send email to nkv@gmail.com with subject'Resume'and body'I will send tommorrow'", additional_kwargs={}, response_metadata={}, id='019961b9-50d5-4a39-92c3-c804be09293a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6xmz6pmgy', 'function': {'arguments': '{"body":"I will send tomorrow","recipient":"nkv@gmail.com","subject":"Resume"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 318, 'total_tokens': 352, 'completion_time': 0.065156331, 'completion_tokens_details': None, 'prompt_time': 0.015634082, 'prompt_tokens_details': None, 'queue_time': 0.161640718, 'total_time': 0.080790413}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eaff7-b86f-75b0-9b91-b8a0a6616f6e-0', tool_calls=[{'name': 'send_email_to